# Students Performance — Exploratory Data Analysis

**Objective:** Perform a structured EDA on the *Students Performance in Exams* dataset — covering data profiling, quality auditing, cleaning, distribution analysis, and insight extraction.

**Workflow:**
1. **Setup & Data Loading** — imports, configuration, and loading the raw CSV
2. **Data Quality Audit & Cleaning** — missing values, duplicates, whitespace, outliers
3. **Feature Engineering** — derived columns for richer analysis
4. **Exploratory Data Analysis** — distributions, correlations, and visual exploration
5. **Insight Deep-Dives** — answer targeted questions with evidence
6. **Summary of Findings**

---
## 1 · Setup & Data Loading
---

### 1.1 · Import Libraries
Load all required libraries. We use **plotly.express** for interactive charts, along with **pandas** and **numpy** for data manipulation.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

### 1.2 · Plotting Defaults
Define a consistent visual style applied to every chart in this notebook.

In [ ]:
# ── Plotting defaults ──────────────────────────────────────────────────────
TEMPLATE = 'plotly_dark'
COLORS = {
    'primary':   '#636EFA',
    'secondary': '#EF553B',
    'accent':    '#00CC96',
    'purple':    '#AB63FA',
    'orange':    '#FFA15A',
    'pink':      '#FF6692',
}
PALETTE_3 = [COLORS['primary'], COLORS['secondary'], COLORS['accent']]


def style(fig, title_x=0.5, **kwargs):
    """Apply a consistent dark-theme style to any Plotly figure."""
    fig.update_layout(template=TEMPLATE, title_x=title_x, **kwargs)
    return fig

### 1.3 · Load the Dataset
Read the CSV and display the first rows to understand the raw structure.

In [ ]:
import os

# Support both local and Kaggle environments
POSSIBLE_PATHS = [
    "students_performance.csv",                          # Local / Colab
    "StudentsPerformance.csv",                           # Alternate local name
    "/kaggle/input/students-performance-in-exams/StudentsPerformance.csv",
]

file_path = next((p for p in POSSIBLE_PATHS if os.path.exists(p)), None)

if file_path is None:
    raise FileNotFoundError(
        "Could not find the dataset CSV. "
        "Place 'students_performance.csv' in the notebook directory."
    )

df = pd.read_csv(file_path)
print(f"Dataset loaded from: {file_path}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

---
## 2 · Data Quality Audit & Cleaning
---

### 2.1 · Data Types & Info
Inspect column types and non-null counts.

In [ ]:
df.info()

### 2.2 · Descriptive Statistics
Numerical and categorical summaries in one view.

In [ ]:
df.describe(include='all').T

### 2.3 · Missing Values

In [ ]:
null_summary = pd.DataFrame({
    'Null Count': df.isnull().sum(),
    'Null %':     (df.isnull().mean() * 100).round(2)
})
print(null_summary)
print(f"\nTotal nulls in dataset: {df.isnull().sum().sum()}")

**Result:** No missing values — the dataset is complete.

### 2.4 · Duplicate Rows

In [ ]:
dup_count = df.duplicated().sum()
print(f"Duplicate rows found: {dup_count}")

if dup_count > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicates detected.")

### 2.5 · Whitespace & Spelling Consistency
Strip leading/trailing whitespace from all text columns and inspect unique values.

In [ ]:
obj_cols = df.select_dtypes(include='object').columns

# Strip whitespace
for col in obj_cols:
    df[col] = df[col].str.strip()

# Inspect unique values
for col in obj_cols:
    print(f"--- {col} ({df[col].nunique()} unique) ---")
    print(sorted(df[col].unique()))
    print()

**Observation:** All categorical values are consistently lowercase with no misspellings.

### 2.6 · Column Standardisation
Rename columns to `snake_case` and verify score dtypes.

In [ ]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('/', '_')
)

SCORE_COLS = ['math_score', 'reading_score', 'writing_score']

for col in SCORE_COLS:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Columns:", list(df.columns))
df.dtypes

---
## 3 · Feature Engineering
---

### 3.1 · Derived Columns
Create `total_score`, `average_score`, `result` (pass/fail), and `grade` (letter).

In [ ]:
def assign_grade(avg):
    """Map an average score to a letter grade."""
    if avg >= 90: return 'A'
    if avg >= 80: return 'B'
    if avg >= 70: return 'C'
    if avg >= 60: return 'D'
    if avg >= 50: return 'E'
    return 'F'


df['total_score']   = df[SCORE_COLS].sum(axis=1)
df['average_score'] = (df['total_score'] / 3).round(2)
df['result']        = np.where(df['average_score'] >= 50, 'Pass', 'Fail')
df['grade']         = df['average_score'].apply(assign_grade)

print("New columns added: total_score, average_score, result, grade")
df.head()

### 3.2 · Outlier Detection & Treatment (IQR)
Identify outliers using the Inter-Quartile Range fence, then cap them (winsorisation).

In [ ]:
def detect_outliers_iqr(series):
    """Return IQR fence boundaries and the outlier mask."""
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    return lower, upper, mask


# Detect
for col in SCORE_COLS:
    lower, upper, mask = detect_outliers_iqr(df[col])
    print(f"{col}:  IQR fence [{lower:.1f}, {upper:.1f}]  |  Outliers: {mask.sum()}")

In [ ]:
# Visualise before capping
fig = px.box(
    df, y=SCORE_COLS,
    title='Box Plots — Score Columns (Before Capping)',
    labels={'value': 'Score', 'variable': 'Subject'},
    color_discrete_sequence=px.colors.qualitative.Set2,
)
style(fig)
fig.show()

In [ ]:
# Cap outliers and recalculate derived columns
for col in SCORE_COLS:
    lower, upper, _ = detect_outliers_iqr(df[col])
    before = ((df[col] < lower) | (df[col] > upper)).sum()
    df[col] = df[col].clip(lower=lower, upper=upper)
    print(f"{col}: capped {before} outliers")

# Recalculate derived columns
df['total_score']   = df[SCORE_COLS].sum(axis=1)
df['average_score'] = (df['total_score'] / 3).round(2)
df['result']        = np.where(df['average_score'] >= 50, 'Pass', 'Fail')
df['grade']         = df['average_score'].apply(assign_grade)

print("\nDerived columns recalculated after capping.")

### 3.3 · Post-Cleaning Verification

In [ ]:
print("=== Post-Cleaning Summary ===")
print(f"Shape           : {df.shape}")
print(f"Nulls remaining : {df.isnull().sum().sum()}")
print(f"Duplicates      : {df.duplicated().sum()}")
print(f"\nDataset is clean and ready for analysis.")

---
## 4 · Exploratory Data Analysis
---

### 4.1 · Score Distributions (Histograms)
Visualise the distribution of each exam score to check for skewness and central tendency.

In [ ]:
for col in SCORE_COLS:
    fig = px.histogram(
        df, x=col, nbins=30, marginal='box',
        title=f'Distribution of {col.replace("_", " ").title()}',
        color_discrete_sequence=[COLORS['primary']],
        opacity=0.75,
    )
    style(fig)
    fig.show()

### 4.2 · Skewness & Kurtosis

In [ ]:
stats_summary = pd.DataFrame({
    'Skewness': [df[c].skew() for c in SCORE_COLS],
    'Kurtosis': [df[c].kurtosis() for c in SCORE_COLS],
}, index=SCORE_COLS).round(3)

stats_summary

### 4.3 · Average Score Distribution

In [ ]:
fig = px.histogram(
    df, x='average_score', nbins=30, marginal='violin',
    title='Distribution of Average Score',
    color_discrete_sequence=[COLORS['secondary']],
    opacity=0.75,
)
style(fig)
fig.show()

### 4.4 · Categorical Distributions
Overview of demographic feature counts.

In [ ]:
# Gender
gender_counts = df['gender'].value_counts().reset_index()
gender_counts.columns = ['gender', 'count']

fig = px.bar(
    gender_counts, x='gender', y='count', color='gender',
    title='Student Count by Gender',
    color_discrete_sequence=[COLORS['primary'], COLORS['secondary']],
    text='count',
)
style(fig, showlegend=False)
fig.update_traces(textposition='outside')
fig.show()

In [ ]:
# Race / Ethnicity
fig = px.pie(
    df, names='race_ethnicity',
    title='Student Distribution by Race / Ethnicity',
    hole=0.45,
    color_discrete_sequence=px.colors.qualitative.Set2,
)
style(fig)
fig.show()

In [ ]:
# Parental Education
EDU_ORDER = [
    'some high school', 'high school', 'some college',
    "associate's degree", "bachelor's degree", "master's degree",
]

edu_counts = (
    df['parental_level_of_education']
    .value_counts()
    .reindex(EDU_ORDER)
    .reset_index()
)
edu_counts.columns = ['education', 'count']

fig = px.bar(
    edu_counts, x='education', y='count', color='education',
    title='Student Count by Parental Education Level',
    color_discrete_sequence=px.colors.sequential.Tealgrn,
    text='count',
)
style(fig, showlegend=False, xaxis_tickangle=-25)
fig.update_traces(textposition='outside')
fig.show()

In [ ]:
# Lunch Type
fig = px.pie(
    df, names='lunch',
    title='Lunch Type Distribution',
    hole=0.45,
    color_discrete_sequence=[COLORS['accent'], COLORS['secondary']],
)
style(fig)
fig.show()

In [ ]:
# Test Preparation Course
prep_counts = df['test_preparation_course'].value_counts().reset_index()
prep_counts.columns = ['test_preparation_course', 'count']

fig = px.bar(
    prep_counts, x='test_preparation_course', y='count',
    color='test_preparation_course',
    title='Test Preparation Course Completion',
    color_discrete_sequence=[COLORS['purple'], COLORS['orange']],
    text='count',
)
style(fig, showlegend=False)
fig.update_traces(textposition='outside')
fig.show()

### 4.5 · Grade & Pass/Fail Distributions

In [ ]:
GRADE_ORDER = ['A', 'B', 'C', 'D', 'E', 'F']
GRADE_COLORS = {
    'A': COLORS['accent'],  'B': COLORS['primary'], 'C': COLORS['orange'],
    'D': COLORS['purple'],  'E': COLORS['secondary'], 'F': COLORS['pink'],
}

grade_counts = df['grade'].value_counts().reindex(GRADE_ORDER).reset_index()
grade_counts.columns = ['grade', 'count']

fig = px.bar(
    grade_counts, x='grade', y='count', color='grade',
    title='Letter Grade Distribution',
    color_discrete_map=GRADE_COLORS,
    text='count',
)
style(fig, showlegend=False)
fig.update_traces(textposition='outside')
fig.show()

In [ ]:
fig = px.pie(
    df, names='result',
    title='Overall Pass / Fail Rate',
    hole=0.5,
    color_discrete_map={'Pass': COLORS['accent'], 'Fail': COLORS['secondary']},
)
style(fig)
fig.show()

### 4.6 · Correlation Heatmap
Assess linear relationships between score columns.

In [ ]:
corr_cols = SCORE_COLS + ['total_score', 'average_score']
corr = df[corr_cols].corr().round(3)

fig = px.imshow(
    corr, text_auto=True,
    title='Correlation Heatmap — Score Columns',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
)
style(fig, width=650, height=550)
fig.show()

### 4.7 · Scatter Matrix — Score Relationships
Pair plot coloured by gender to visualise inter-score relationships.

In [ ]:
fig = px.scatter_matrix(
    df, dimensions=SCORE_COLS, color='gender',
    title='Scatter Matrix — Scores by Gender',
    color_discrete_sequence=[COLORS['primary'], COLORS['secondary']],
    opacity=0.5,
)
style(fig, width=800, height=700)
fig.show()

### 4.8 · Sunburst — Multi-Level Breakdown
A hierarchical view: Gender → Lunch → Test Prep, sized by student count.

In [ ]:
fig = px.sunburst(
    df, path=['gender', 'lunch', 'test_preparation_course'],
    title='Sunburst — Gender → Lunch → Test Prep',
    color_discrete_sequence=px.colors.qualitative.Pastel,
)
style(fig)
fig.show()

---
## 5 · Insight Deep-Dives
---

### Helper: Student Profile Function
Reusable function to avoid duplicating profiling code for top/bottom students.

In [ ]:
def profile_segment(segment_df, label="Segment"):
    """Print a demographic profile of a student segment."""
    profile_cols = [
        ('gender', 'Gender'),
        ('lunch', 'Lunch Type'),
        ('test_preparation_course', 'Test Prep'),
        ('parental_level_of_education', 'Parental Education'),
        ('race_ethnicity', 'Race/Ethnicity'),
    ]
    print(f"--- {label} ({len(segment_df)} students) ---")
    for col, name in profile_cols:
        pct = segment_df[col].value_counts(normalize=True).mul(100).round(1)
        print(f"\n{name}:\n{pct}")

### Q1 — Do females outperform males in specific subjects?
Compare mean scores per gender across subjects.

In [ ]:
gender_means = df.groupby('gender')[SCORE_COLS + ['average_score']].mean().round(2)
print(gender_means)
print('\nDifference (Female − Male):')
print((gender_means.loc['female'] - gender_means.loc['male']).round(2))

In [ ]:
gm = gender_means.reset_index().melt(
    id_vars='gender', var_name='subject', value_name='mean_score'
)
fig = px.bar(
    gm, x='subject', y='mean_score', color='gender',
    barmode='group',
    title='Q1 — Mean Scores by Gender',
    color_discrete_sequence=[COLORS['secondary'], COLORS['primary']],
    text='mean_score',
)
style(fig)
fig.update_traces(textposition='outside')
fig.show()

**Insight Q1:**
- **Males outperform females in Math** by a notable margin.
- **Females outperform males in both Reading and Writing** — the gap is especially large in Writing.
- Overall, female students have a slightly higher average score, driven by their literacy advantage.

### Q2 — Does parental education level affect student performance?
Group average scores by parental education and look for a trend.

In [ ]:
edu_avg = (
    df.groupby('parental_level_of_education')['average_score']
    .mean()
    .reindex(EDU_ORDER)
    .round(2)
)
print(edu_avg)

fig = px.line(
    x=edu_avg.index, y=edu_avg.values,
    markers=True,
    title='Q2 — Average Score by Parental Education Level',
    labels={'x': 'Parental Education', 'y': 'Average Score'},
)
style(fig, xaxis_tickangle=-25)
fig.update_traces(line_color=COLORS['accent'], marker_size=10)
fig.show()

**Insight Q2:**
- Clear **positive trend**: higher parental education → higher student scores.
- Students whose parents hold a **master's degree** score the highest.
- The jump from "some high school" to "bachelor's/master's degree" is substantial.

### Q3 — Does completing the test preparation course improve scores?
Compare means and visualise distributions for completed vs. none.

In [ ]:
prep_means = df.groupby('test_preparation_course')[SCORE_COLS + ['average_score']].mean().round(2)
print(prep_means)
print('\nDifference (Completed − None):')
print((prep_means.loc['completed'] - prep_means.loc['none']).round(2))

In [ ]:
# Grouped bar chart
pm = prep_means.reset_index().melt(
    id_vars='test_preparation_course',
    var_name='subject', value_name='mean_score',
)
fig = px.bar(
    pm, x='subject', y='mean_score', color='test_preparation_course',
    barmode='group',
    title='Q3 — Mean Scores: Prep Course Completed vs. None',
    color_discrete_sequence=[COLORS['accent'], COLORS['secondary']],
    text='mean_score',
)
style(fig)
fig.update_traces(textposition='outside')
fig.show()

In [ ]:
# Violin plots for distributional comparison
df_melted = df.melt(
    id_vars=['test_preparation_course'], value_vars=SCORE_COLS,
    var_name='subject', value_name='score',
)
fig = px.violin(
    df_melted, x='subject', y='score', color='test_preparation_course',
    box=True,
    title='Q3 — Score Distributions by Test Preparation Course',
    color_discrete_sequence=[COLORS['purple'], COLORS['accent']],
)
style(fig)
fig.show()

**Insight Q3:**
- Students who **completed the prep course** scored higher in **all three subjects**.
- The largest improvement is in **Writing**, followed by Reading and Math.
- The prep course is effective and should be encouraged for all students.

### Q4 — Does lunch type impact performance?
Lunch type is often used as a proxy for socio-economic status.

In [ ]:
lunch_means = df.groupby('lunch')[SCORE_COLS + ['average_score']].mean().round(2)
print(lunch_means)
print('\nDifference (Standard − Free/Reduced):')
print((lunch_means.loc['standard'] - lunch_means.loc['free/reduced']).round(2))

In [ ]:
fig = px.box(
    df, x='lunch', y='average_score', color='lunch',
    title='Q4 — Average Score Distribution by Lunch Type',
    color_discrete_sequence=[COLORS['secondary'], COLORS['accent']],
    points='all',
)
style(fig, showlegend=False)
fig.show()

**Insight Q4:**
- Students on **standard lunch** consistently score **~10–12 points higher** than those on free/reduced lunch.
- This is the **single largest performance gap** in the dataset, suggesting socio-economic factors play a major role.

### Q5 — Which racial/ethnic group performs highest and lowest?
Rank groups by mean scores across all subjects.

In [ ]:
race_perf = (
    df.groupby('race_ethnicity')[SCORE_COLS + ['average_score']]
    .mean()
    .round(2)
    .sort_values('average_score', ascending=False)
)
print(race_perf)

In [ ]:
rp = race_perf.reset_index().melt(
    id_vars='race_ethnicity', value_vars=SCORE_COLS,
    var_name='subject', value_name='mean_score',
)
fig = px.bar(
    rp, x='race_ethnicity', y='mean_score', color='subject',
    barmode='group',
    title='Q5 — Mean Scores by Race / Ethnicity',
    color_discrete_sequence=PALETTE_3,
)
style(fig)
fig.show()

**Insight Q5:**
- **Group E** consistently achieves the highest average scores across all three subjects.
- **Group A** has the lowest average scores.
- The gap is moderate but consistent.

### Q6 — What is the relationship between Math, Reading, and Writing scores?
Are students who are strong in one subject also strong in others?

In [ ]:
print('Correlation Matrix:')
print(df[SCORE_COLS].corr().round(3))

In [ ]:
fig = px.scatter(
    df, x='reading_score', y='writing_score', color='gender',
    trendline='ols',
    title='Q6 — Reading vs. Writing (Strongest Correlation)',
    color_discrete_sequence=[COLORS['primary'], COLORS['secondary']],
    opacity=0.6,
)
style(fig)
fig.show()

In [ ]:
fig = px.scatter(
    df, x='math_score', y='reading_score', color='gender',
    trendline='ols',
    title='Q6 — Math vs. Reading',
    color_discrete_sequence=[COLORS['primary'], COLORS['secondary']],
    opacity=0.6,
)
style(fig)
fig.show()

**Insight Q6:**
- **Reading and Writing** have the **strongest correlation** (~0.95) — students who read well almost always write well.
- **Math** is also positively correlated with the other two, but weaker (~0.8).
- Verbal/literacy skills are closely linked, while mathematical ability is somewhat independent.

### Q7 & Q8 — What characterises the top and bottom 10% of students?
Profile the highest- and lowest-performing students side by side.

In [ ]:
top_threshold = df['average_score'].quantile(0.90)
bottom_threshold = df['average_score'].quantile(0.10)

top_students = df[df['average_score'] >= top_threshold]
bottom_students = df[df['average_score'] <= bottom_threshold]

print(f"Top 10% threshold:    average_score >= {top_threshold}")
print(f"Bottom 10% threshold: average_score <= {bottom_threshold}\n")

profile_segment(top_students, "Top 10%")
print("\n" + "=" * 50 + "\n")
profile_segment(bottom_students, "Bottom 10%")

**Insight Q7 (Top 10%):**
- More **females** than males — reflecting the reading/writing advantage.
- **Standard lunch** is heavily overrepresented.
- Higher proportion **completed the test preparation course**.
- Parents with **bachelor's or master's degrees** are disproportionately represented.
- **Group E** appears more frequently.

**Insight Q8 (Bottom 10%):**
- More **males** than females.
- **Free/reduced lunch** is heavily overrepresented.
- Vast majority **did NOT complete** the test preparation course.
- Parents with **"some high school" or "high school"** education dominate.

### Q9 — Is there a combined effect of lunch type + test prep?
Examine the interaction between these two key factors.

In [ ]:
combo = (
    df.groupby(['lunch', 'test_preparation_course'])['average_score']
    .mean()
    .round(2)
    .reset_index()
)
print(combo)

fig = px.bar(
    combo, x='lunch', y='average_score', color='test_preparation_course',
    barmode='group',
    title='Q9 — Average Score: Lunch × Test Preparation',
    color_discrete_sequence=[COLORS['accent'], COLORS['secondary']],
    text='average_score',
)
style(fig)
fig.update_traces(textposition='outside')
fig.show()

**Insight Q9:**
- **Standard lunch + Completed prep course** is the highest-performing combination.
- **Free/reduced lunch + No prep course** is the lowest-performing combination.
- The prep course can **partially offset** socio-economic disadvantage.

### Q10 — Does gender interact with test preparation course effectiveness?
Check if the prep course benefits males and females equally.

In [ ]:
gender_prep = (
    df.groupby(['gender', 'test_preparation_course'])['average_score']
    .mean()
    .round(2)
    .reset_index()
)
print(gender_prep)

fig = px.bar(
    gender_prep, x='gender', y='average_score', color='test_preparation_course',
    barmode='group',
    title='Q10 — Average Score: Gender × Test Preparation',
    color_discrete_sequence=[COLORS['accent'], COLORS['purple']],
    text='average_score',
)
style(fig)
fig.update_traces(textposition='outside')
fig.show()

**Insight Q10:**
- The test preparation course **benefits both genders** roughly equally.
- **Female students who completed the course** have the highest overall average of any subgroup.
- The prep course is universally beneficial regardless of gender.

---
## 6 · Summary of Key Findings

| # | Finding |
|---|---------|
| 1 | **No data quality issues** — zero nulls, zero duplicates, no whitespace or spelling problems. |
| 2 | **Gender gap** — Males lead in Math; Females lead in Reading and Writing. |
| 3 | **Parental education matters** — Higher parental education → higher student scores (linear trend). |
| 4 | **Test prep course works** — Students who completed it scored 5–8 points higher across all subjects. |
| 5 | **Lunch type = strongest predictor** — Standard lunch students outperform free/reduced by ~10–12 points (socio-economic proxy). |
| 6 | **Group E** tops all racial/ethnic groups; **Group A** scores lowest. |
| 7 | **Reading & Writing** are near-perfectly correlated (r ≈ 0.95); Math is related but more independent. |
| 8 | **Top 10%** are predominantly female, standard lunch, prep-course completers with highly educated parents. |
| 9 | **Bottom 10%** are predominantly male, free/reduced lunch, no prep course, with lower parental education. |
| 10 | **Combined effect** — Standard lunch + prep course is the best combo; the prep course can partially offset low socio-economic status. |

---
*Analysis completed. Dataset is clean and all insights are documented.*